# Phase 8: Product Analysis (ABC / Pareto)

Classifies all 1,615 products into A/B/C revenue tiers and identifies the small set of products driving most of the business.

## Setup

In [1]:
import pandas as pd, numpy as np, json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

df = pd.read_csv('../data/polokwane_sales_clean.csv', parse_dates=['date'])
sales = df[df['value_zar'] > 0].copy()
out = '../outputs'

prod = sales.groupby('product_desc').agg(
    revenue=('value_zar','sum'), qty=('qty','sum'), mass_kg=('mass_kg','sum'),
    transactions=('value_zar','count')
).sort_values('revenue', ascending=False)

prod['cum_revenue'] = prod['revenue'].cumsum()
prod['cum_pct'] = prod['cum_revenue'] / prod['revenue'].sum() * 100
prod['rank'] = range(1, len(prod)+1)

def abc_class(pct):
    if pct <= 80: return 'A'
    elif pct <= 95: return 'B'
    else: return 'C'
prod['abc_class'] = prod['cum_pct'].apply(abc_class)

abc_summary = prod.groupby('abc_class').agg(products=('revenue','count'), revenue=('revenue','sum'))
abc_summary['pct_of_products'] = (abc_summary['products']/len(prod)*100).round(1)
abc_summary['pct_of_revenue'] = (abc_summary['revenue']/prod['revenue'].sum()*100).round(1)
print("ABC classification:")
print(abc_summary)
abc_summary.to_csv(f'{out}/product_abc_summary.csv')
prod.to_csv(f'{out}/product_abc_full.csv')

# Pareto chart
fig, ax1 = plt.subplots(figsize=(10,5))
top50 = prod.head(50)
ax1.bar(range(len(top50)), top50['revenue'], color='#2563eb')
ax1.set_ylabel('Revenue (ZAR)', color='#2563eb')
ax1.set_xlabel('Product rank (top 50)')
ax2 = ax1.twinx()
ax2.plot(range(len(top50)), top50['cum_pct'], color='#dc2626', linewidth=2)
ax2.axhline(80, color='gray', linestyle='--', linewidth=1)
ax2.set_ylabel('Cumulative % of revenue', color='#dc2626')
plt.title('Pareto Analysis — Top 50 Products by Revenue')
plt.tight_layout()
plt.savefig(f'{out}/product_pareto.png', dpi=150)
plt.close()

n_a = (prod['abc_class']=='A').sum()
print(f"\n{n_a} products ({n_a/len(prod):.1%} of the {len(prod)} product range) drive 80% of revenue.")

# Slow movers: bottom decile by revenue but still actively sold
slow = prod[prod['abc_class']=='C'].sort_values('transactions', ascending=False).head(15)
slow.to_csv(f'{out}/slow_movers_sample.csv')

summary = {
    'total_products': int(len(prod)),
    'class_A_products': int(n_a),
    'class_A_pct_of_products': float(n_a/len(prod)*100),
    'top_product': prod.index[0],
    'top_product_revenue': float(prod.iloc[0]['revenue']),
    'top10_products_revenue_share': float(prod.head(10)['revenue'].sum()/prod['revenue'].sum()*100),
}
with open(f'{out}/product_analysis_summary.json','w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

ABC classification:
           products      revenue  pct_of_products  pct_of_revenue
abc_class                                                        
A               155  78509346.77              9.6            79.9
B               299  14852336.62             18.5            15.1
C              1161   4920900.39             71.9             5.0



155 products (9.6% of the 1615 product range) drive 80% of revenue.
{
  "total_products": 1615,
  "class_A_products": 155,
  "class_A_pct_of_products": 9.597523219814242,
  "top_product": "Shoulder Ribs Smoked P/Kg",
  "top_product_revenue": 5059321.67,
  "top10_products_revenue_share": 28.427324328937175
}
